# Import Packages

In [194]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import talib.abstract as ta
import scipy as sp
from sklearn.preprocessing import PowerTransformer
from typing import List
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import backtesting

# Get Data

In [195]:
# Apple Futures data using yfinance
ticker = 'MSFT' #'ETH-USD'
data = yf.download(ticker, start="2015-01-01", interval="1D")

# Check the data
display(data.head())

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT
Date,,,,,
2015-01-02,39.767689,40.328995,39.580589,39.682644,27913900
2015-01-05,39.401981,39.742165,39.333943,39.435997,39673900
2015-01-06,38.823673,39.759182,38.730122,39.444511,36447900
2015-01-07,39.316933,39.512539,38.687591,39.104317,29114100
2015-01-08,40.473572,40.609646,39.733669,39.759182,29645200


In [196]:
if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)

In [197]:
data.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,39.767689,40.328995,39.580589,39.682644,27913900
2015-01-05,39.401981,39.742165,39.333943,39.435997,39673900
2015-01-06,38.823673,39.759182,38.730122,39.444511,36447900
2015-01-07,39.316933,39.512539,38.687591,39.104317,29114100
2015-01-08,40.473572,40.609646,39.733669,39.759182,29645200


In [198]:
data_copy = data.copy()

# Preprocess

## Indicators

### RSI

https://github.com/TA-Lib/ta-lib-python/issues/448

In [199]:
def calculate_rsi(series, periods=14):
    """Wilder's Exponential Smoothing ile RSI hesaplaması."""
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.ewm(alpha=1/periods, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/periods, adjust=False).mean()
    
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

In [200]:
data_copy['RSI'] = ta.RSI(data_copy['Close'], timeperiod = 14)
#data_copy['RSI'] = calculate_rsi(data_copy['Close'], periods=14)

### SMA

In [201]:
data_copy['SMA_Short'] = ta.SMA(data_copy['Close'], timeperiod = 20)
data_copy['SMA_Long'] = ta.SMA(data_copy['Close'], timeperiod = 50)

In [202]:
plot_df = data_copy.tail(400).copy()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.03, 
                        subplot_titles=(f'{ticker} Günlük Grafik ve Gap Sinyalleri', 'İşlem Hacmi'), 
                        row_width=[0.2, 0.7])
fig.add_trace(go.Candlestick(x=plot_df.index,
                                open=plot_df['Open'], high=plot_df['High'],
                                low=plot_df['Low'], close=plot_df['Close'],
                                name='Fiyat'), row=1, col=1)

# 2. Kısa ve Uzun Hareketli Ortalamalar (Trend tespiti için)
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_Short'], 
                            mode='lines', line=dict(color='blue', width=1, dash='dot'), 
                            name='SMA Kısa'), row=1, col=1)

fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_Long'], 
                            mode='lines', line=dict(color='purple', width=1, dash='dot'), 
                            name='SMA Kısa'), row=1, col=1)


fig.update_layout(
    xaxis_rangeslider_visible=False, # Alttaki gereksiz kaydırma çubuğunu gizle
    height=800, 
    template='plotly_white',
    hovermode='x unified', # Tek bir tarihe gelince tüm verileri (Açılış, kapanış, hacim vs) göster
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Y ekseni isimleri
fig.update_yaxes(title_text="Fiyat ($)", row=1, col=1)
fig.update_yaxes(title_text="Hacim", row=2, col=1)

fig.show()

### Volume SMA

In [203]:
data_copy["Vol_SMA"] = ta.SMA(data_copy['Volume'], 20)

### Support & Resistance Lines

In [204]:
# Destek/Direnç tespiti için geçmiş N günün ekstrem değerleri (bugünü shift(1) ile hariç tutuyoruz)
data_copy['Range_High'] = data_copy['High'].rolling(window=20).max().shift(1)
data_copy['Range_Low'] = data_copy['Low'].rolling(window=20).min().shift(1)


In [205]:
plot_df = data_copy.tail(400).copy()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.03, 
                        subplot_titles=(f'{ticker} Günlük Grafik ve Gap Sinyalleri', 'İşlem Hacmi'), 
                        row_width=[0.2, 0.7])
fig.add_trace(go.Candlestick(x=plot_df.index,
                                open=plot_df['Open'], high=plot_df['High'],
                                low=plot_df['Low'], close=plot_df['Close'],
                                name='Fiyat'), row=1, col=1)

# 2. Kısa ve Uzun Hareketli Ortalamalar (Trend tespiti için)
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Range_High'], 
                            mode='lines', line=dict(color='red', width=1, dash='dot'), 
                            name='SMA Kısa'), row=1, col=1)

fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Range_Low'], 
                            mode='lines', line=dict(color='green', width=1, dash='dot'), 
                            name='SMA Kısa'), row=1, col=1)


fig.update_layout(
    xaxis_rangeslider_visible=False, # Alttaki gereksiz kaydırma çubuğunu gizle
    height=800, 
    template='plotly_white',
    hovermode='x unified', # Tek bir tarihe gelince tüm verileri (Açılış, kapanış, hacim vs) göster
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Y ekseni isimleri
fig.update_yaxes(title_text="Fiyat ($)", row=1, col=1)
fig.update_yaxes(title_text="Hacim", row=2, col=1)

fig.show()

### ADX

In [206]:
data_copy["ADX"] = ta.ADX(data_copy['High'], data_copy['Low'], data_copy['Close'], timeperiod=14)

## Conditions

### Gap Detection

In [207]:
gap_up = data_copy['Low'] > data_copy['High'].shift(1)
gap_down = data_copy['High'] < data_copy['Low'].shift(1)

### Exhaustion 

In [208]:
exhaustion_gap_up = data_copy['Open'] > data_copy['Close'].shift(1)
exhaustion_gap_down = data_copy['Open'] < data_copy['Close'].shift(1)

### Volume Filter

In [209]:
high_vol = data_copy['Volume'] > (1.5 * data_copy['Vol_SMA'])
extreme_vol  = data_copy['Volume'] > (2 * data_copy['Vol_SMA'])

### Breakout Up & Down

In [210]:
breakout_up = data_copy['Low'] > data_copy['Range_High']
breakout_down = data_copy['High'] < data_copy['Range_Low']

### Consolidation

In [211]:
consolidation = data_copy["ADX"] < 25

### Bullish & Bearish

In [212]:
uptrend = data_copy['SMA_Short'] > data_copy['SMA_Long']
downtrend = data_copy['SMA_Short'] < data_copy['SMA_Long']

### Overbought & Oversold

In [213]:
overbought= data_copy['RSI'] > 70
oversold= data_copy['RSI'] < 30

### Conditions List

In [214]:
conditions = [
        # Breakaway: Direnç/Destek kırılımı + Yüksek Hacim
        (gap_up & breakout_up & high_vol & consolidation.shift(1)),
        (gap_down & breakout_down & high_vol & consolidation.shift(1)),
        
        # Exhaustion: Trend içi aşırı bölge + Ekstrem Hacim
        (exhaustion_gap_up & uptrend & overbought & extreme_vol & ~consolidation),
        (exhaustion_gap_down & downtrend & oversold & extreme_vol & ~consolidation),
        
        # Runaway: Trend içi gap + Kırılım yok + Aşırı bölgede değil
        (gap_up & uptrend & ~breakout_up & ~overbought & ~consolidation),
        (gap_down & downtrend & ~breakout_down & ~oversold & ~consolidation),
        
        # Common: Range içi, düşük hacimli (yukarıdaki şartları sağlamayan)
        (gap_up & ~breakout_up & ~high_vol),
        (gap_down & ~breakout_down & ~high_vol)
    ]

# Classification

In [215]:
choices = [
    'Breakaway Up', 'Breakaway Down',
    'Exhaustion Up', 'Exhaustion Down',
    'Runaway Up', 'Runaway Down',
    'Common Up', 'Common Down'
]

data_copy['Gap_Type'] = np.select(conditions, choices, default='None')

# Visualization

In [216]:
def visualize_gaps(df, ticker="AAPL", lookback=300):
    """
    Sınıflandırılmış Gap verilerini interaktif Plotly grafiği ile görselleştirir.
    Görsel karmaşayı önlemek için varsayılan olarak son 300 günü (lookback) çizer.
    """
    # Grafiği kalabalıklaştırmamak adına verinin son kısmını alıyoruz
    plot_df = df.tail(lookback).copy()
    
    # 2 satırlı bir figür oluştur: Üstte Fiyat ve Sinyaller, Altta Hacim
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.03, 
                        subplot_titles=(f'{ticker} Günlük Grafik ve Gap Sinyalleri', 'İşlem Hacmi'), 
                        row_width=[0.2, 0.7])

    # 1. Mum Grafik (Candlestick)
    fig.add_trace(go.Candlestick(x=plot_df.index,
                                 open=plot_df['Open'], high=plot_df['High'],
                                 low=plot_df['Low'], close=plot_df['Close'],
                                 name='Fiyat'), row=1, col=1)

    # 2. Kısa ve Uzun Hareketli Ortalamalar (Trend tespiti için)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_Short'], 
                             mode='lines', line=dict(color='blue', width=1, dash='dot'), 
                             name='SMA Kısa'), row=1, col=1)
    
    # 3. Gap Sinyallerini İşaretleme Sözlüğü (Renk ve Semboller)
    gap_styles = {
        'Breakaway Up': {'color': 'darkgreen', 'symbol': 'triangle-up', 'size': 14},
        'Breakaway Down': {'color': 'darkred', 'symbol': 'triangle-down', 'size': 14},
        'Runaway Up': {'color': 'lightgreen', 'symbol': 'arrow-up', 'size': 10},
        'Runaway Down': {'color': 'pink', 'symbol': 'arrow-down', 'size': 10},
        'Exhaustion Up': {'color': 'orange', 'symbol': 'diamond-tall', 'size': 20},
        'Exhaustion Down': {'color': 'purple', 'symbol': 'diamond-tall', 'size': 20},
        'Common Up': {'color': 'gray', 'symbol': 'circle-open', 'size': 8},
        'Common Down': {'color': 'gray', 'symbol': 'circle-open', 'size': 8}
    }

    # Sinyalleri grafiğe ekleme
    for gap_type, style in gap_styles.items():
        gap_data = plot_df[plot_df['Gap_Type'] == gap_type]
        if not gap_data.empty:
            # İşaretçiler (marker) mumların içine girmesin diye yukarı gapleri aşağıya, aşağı gapleri yukarıya koyuyoruz
            y_pos = gap_data['Low'] * 0.98 if 'Up' in gap_type else gap_data['High'] * 1.02
            
            fig.add_trace(go.Scatter(x=gap_data.index, y=y_pos,
                                     mode='markers',
                                     marker=dict(symbol=style['symbol'], 
                                                 size=style['size'], 
                                                 color=style['color'], 
                                                 line=dict(width=1, color='black')),
                                     name=gap_type), row=1, col=1)

    # 4. Hacim Barları (Volume)
    # Düşüş günleri kırmızı, yükseliş günleri yeşil hacim barı
    volume_colors = ['red' if row['Open'] > row['Close'] else 'green' for index, row in plot_df.iterrows()]
    fig.add_trace(go.Bar(x=plot_df.index, y=plot_df['Volume'], 
                         marker_color=volume_colors, name='Hacim'), row=2, col=1)
    
    # Hacim Hareketli Ortalaması (Breakaway ve Exhaustion tespiti için kritik)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Vol_SMA'], 
                             mode='lines', line=dict(color='black', width=2), 
                             name='Hacim SMA'), row=2, col=1)
    
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Range_High'], 
                                mode='lines', line=dict(color='red', width=1, dash='dot'), 
                                name='SMA Kısa'), row=1, col=1)

    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['Range_Low'], 
                                mode='lines', line=dict(color='green', width=1, dash='dot'), 
                                name='SMA Kısa'), row=1, col=1)

    # Layout Ayarları
    fig.update_layout(
        xaxis_rangeslider_visible=False, # Alttaki gereksiz kaydırma çubuğunu gizle
        height=800, 
        template='plotly_white',
        hovermode='x unified', # Tek bir tarihe gelince tüm verileri (Açılış, kapanış, hacim vs) göster
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    # Y ekseni isimleri
    fig.update_yaxes(title_text="Fiyat ($)", row=1, col=1)
    fig.update_yaxes(title_text="Hacim", row=2, col=1)

    fig.show()

In [217]:
visualize_gaps(data_copy, ticker=ticker, lookback=1000)

# Backtesting

In [221]:
import vectorbt as vbt
import pandas as pd

def run_gap_backtest(df, init_cash=10000, fees=0.001):
    """
    Sınıflandırılmış Gap verileri üzerinden VectorBT ile backtest simülasyonu çalıştırır.
    """
    # 1. Alış (Long) ve Satış (Short/Exit) Maskelerini Oluşturma
    entries = df['Gap_Type'].isin(['Breakaway Up', 'Runaway Up', 'Exhaustion Down'])
    exits = df['Gap_Type'].isin(['Breakaway Down', 'Runaway Down', 'Exhaustion Up'])

    # HATA ÇÖZÜMÜ: Alt seviye Numba çağrısı yerine .vbt metodunu kullanıyoruz.
    # Bu metot çakışan sinyalleri (aynı gün hem al hem sat) otomatik temizler.
    entries, exits = entries.vbt.signals.clean(exits)

    # 2. Portföy Simülasyonunu Başlatma
    portfolio = vbt.Portfolio.from_signals(
        df['Close'],            
        entries=entries,        
        exits=exits,            
        init_cash=init_cash,    
        fees=fees,              
        direction='both',       # Hem Long hem Short (Açığa Satış)
        freq='1D'               
    )

    # 3. Performans Metriklerini Çıkarma
    print("--- BACKTEST SONUÇLARI ---")
    print(portfolio.stats())
    
    # Kümülatif Getiri grafiğini çizdirmek istersen aşağıdaki satırı aktif edebilirsin
    # portfolio.plot().show()
    
    return portfolio

# -- ÇALIŞTIRMA --
pf_results = run_gap_backtest(data_copy)

--- BACKTEST SONUÇLARI ---
Start                         2015-01-02 00:00:00
End                           2026-04-17 00:00:00
Period                         2839 days 00:00:00
Start Value                               10000.0
End Value                            -6841.538113
Total Return [%]                      -168.415381
Benchmark Return [%]                   963.149561
Max Gross Exposure [%]                1138.358219
Total Fees Paid                        121.028633
Max Drawdown [%]                       173.743314
Max Drawdown Duration          1895 days 00:00:00
Total Trades                                    7
Total Closed Trades                             6
Total Open Trades                               1
Open Trade PnL                       -5306.638464
Win Rate [%]                            33.333333
Best Trade [%]                          54.215397
Worst Trade [%]                       -134.591334
Avg Winning Trade [%]                    38.79161
Avg Losing Trade [%]   

In [228]:
import vectorbt as vbt
import pandas as pd

def run_gap_backtest_optimized(df, init_cash=10000, fees=0.001):
    
    # 1. Sinyalleri Al
    entries = df['Gap_Type'].isin(['Breakaway Up', 'Runaway Up', 'Exhaustion Down'])
    
    # KISA POZİSYONLARI ŞİMDİLİK DEVRE DIŞI BIRAKIYORUZ (Sadece Long Testi)
    exits = df['Gap_Type'].isin(['Breakaway Down','Exhaustion Up'])
    entries, exits = entries.vbt.signals.clean(exits)

    # 2. Risk Yönetimli Portföy
    portfolio = vbt.Portfolio.from_signals(
        df['Close'],            
        entries=entries,        
        exits=exits,            # Zıt sinyalle çıkışı kapattık, artık TP/SL ile çıkacak
        init_cash=init_cash,    
        fees=fees,              
        direction='longonly',   # Sadece hisse alım yönlü test
        freq='1D'               
    )

    print("--- OPTİMİZE EDİLMİŞ BACKTEST ---")
    print(portfolio.stats())
    
    return portfolio

pf_results = run_gap_backtest_optimized(data_copy)

--- OPTİMİZE EDİLMİŞ BACKTEST ---
Start                         2015-01-02 00:00:00
End                           2026-04-17 00:00:00
Period                         2839 days 00:00:00
Start Value                               10000.0
End Value                             33545.87877
Total Return [%]                       235.458788
Benchmark Return [%]                   963.149561
Max Gross Exposure [%]                      100.0
Total Fees Paid                        342.032078
Max Drawdown [%]                        37.043885
Max Drawdown Duration           526 days 00:00:00
Total Trades                                    8
Total Closed Trades                             7
Total Open Trades                               1
Open Trade PnL                       -9328.879628
Win Rate [%]                            71.428571
Best Trade [%]                          61.674443
Worst Trade [%]                         -31.12368
Avg Winning Trade [%]                   48.462066
Avg Losing Trade

In [231]:
pf_results.stats()['Total Return [%]']

np.float64(235.45878769861832)

# More than 1 ticker

In [192]:
import sys
import os
import importlib


sys.path.append(os.path.abspath(".."))  # bir üst klasör
import src.indicators
import src.conditions 
import src.utils
importlib.reload(src.utils)
import src.strategy
importlib.reload(src.indicators)  
importlib.reload(src.conditions)
importlib.reload(src.strategy)
from src.indicators import Indicators
from src.conditions import Conditions
from src.strategy import Strategy

ModuleNotFoundError: No module named 'utils'

In [174]:
# Apple Futures data using yfinance
ticker = ['MSFT','AAPL'] #'ETH-USD'
data = yf.download(ticker, start="2015-01-01", interval="1D")

# Check the data
display(data.head())

[*********************100%***********************]  2 of 2 completed


Price           Close                  High                   Low             \
Ticker           AAPL       MSFT       AAPL       MSFT       AAPL       MSFT   
Date                                                                           
2015-01-02  24.214888  39.767704  24.682220  40.329010  23.776348  39.580604   
2015-01-05  23.532721  39.401993  24.064284  39.742176  23.346674  39.333954   
2015-01-06  23.534935  38.823673  23.794071  39.759182  23.173914  38.730122   
2015-01-07  23.864950  39.316952  23.964618  39.512558  23.632391  38.687610   
2015-01-08  24.781898  40.473568  24.839485  40.609642  24.075362  39.733665   

Price            Open                Volume            
Ticker           AAPL       MSFT       AAPL      MSFT  
Date                                                   
2015-01-02  24.671145  39.682659  212818400  27913900  
2015-01-05  23.984549  39.436009  257142000  39673900  
2015-01-06  23.596950  39.444511  263188400  36447900  
2015-01-07  23.743133  39.104336  160423600  29114100  
2015-01-08  24.192751  39.759178  237458000  29645200

In [175]:
data.columns

MultiIndex([( 'Close', 'AAPL'),
            ( 'Close', 'MSFT'),
            (  'High', 'AAPL'),
            (  'High', 'MSFT'),
            (   'Low', 'AAPL'),
            (   'Low', 'MSFT'),
            (  'Open', 'AAPL'),
            (  'Open', 'MSFT'),
            ('Volume', 'AAPL'),
            ('Volume', 'MSFT')],
           names=['Price', 'Ticker'])

In [176]:
[ ' '.join(col_tuple) for col_tuple in data.columns.to_flat_index()]

['Close AAPL',
 'Close MSFT',
 'High AAPL',
 'High MSFT',
 'Low AAPL',
 'Low MSFT',
 'Open AAPL',
 'Open MSFT',
 'Volume AAPL',
 'Volume MSFT']

In [177]:
if isinstance(data.columns, pd.MultiIndex):
        data.columns = [ ' '.join(col_tuple) for col_tuple in data.columns.to_flat_index()]

In [178]:
display(data.head())

,Close AAPL,Close MSFT,High AAPL,High MSFT,Low AAPL,Low MSFT,Open AAPL,Open MSFT,Volume AAPL,Volume MSFT
Date,,,,,,,,,,
2015-01-02,24.214888,39.767704,24.682220,40.329010,23.776348,39.580604,24.671145,39.682659,212818400,27913900
2015-01-05,23.532721,39.401993,24.064284,39.742176,23.346674,39.333954,23.984549,39.436009,257142000,39673900
2015-01-06,23.534935,38.823673,23.794071,39.759182,23.173914,38.730122,23.596950,39.444511,263188400,36447900
2015-01-07,23.864950,39.316952,23.964618,39.512558,23.632391,38.687610,23.743133,39.104336,160423600,29114100
2015-01-08,24.781898,40.473568,24.839485,40.609642,24.075362,39.733665,24.192751,39.759178,237458000,29645200


In [179]:
obj_ind = Indicators(data=data)
data_new = obj_ind.get_calculated_data()

In [180]:
data_new.columns

Index(['Close AAPL', 'Close MSFT', 'High AAPL', 'High MSFT', 'Low AAPL',
       'Low MSFT', 'Open AAPL', 'Open MSFT', 'Volume AAPL', 'Volume MSFT',
       'AAPL_RSI', 'AAPL_SMA_Short', 'AAPL_SMA_Long', 'MSFT_RSI',
       'MSFT_SMA_Short', 'MSFT_SMA_Long', 'AAPL_Range_High', 'AAPL_Range_Low',
       'AAPL_ADX', 'MSFT_Range_High', 'MSFT_Range_Low', 'MSFT_ADX',
       'AAPL_Vol_SMA', 'MSFT_Vol_SMA'],
      dtype='object')

In [181]:
[ i.lstrip('Close ') for i in data_new.columns if 'Close' in i ]

['AAPL', 'MSFT']

# Backtesting 

In [184]:
conditioner = Conditions(data_new)

In [186]:
data_new_ = conditioner.set_conditions()

In [187]:
data_new.columns

Index(['Close AAPL', 'Close MSFT', 'High AAPL', 'High MSFT', 'Low AAPL',
       'Low MSFT', 'Open AAPL', 'Open MSFT', 'Volume AAPL', 'Volume MSFT',
       'AAPL_RSI', 'AAPL_SMA_Short', 'AAPL_SMA_Long', 'MSFT_RSI',
       'MSFT_SMA_Short', 'MSFT_SMA_Long', 'AAPL_Range_High', 'AAPL_Range_Low',
       'AAPL_ADX', 'MSFT_Range_High', 'MSFT_Range_Low', 'MSFT_ADX',
       'AAPL_Vol_SMA', 'MSFT_Vol_SMA', 'AAPL_gap_up', 'AAPL_gap_down',
       'AAPL_exhaustion_gap_up', 'AAPL_exhaustion_gap_down', 'AAPL_high_vol',
       'AAPL_extreme_vol', 'AAPL_breakout_up', 'AAPL_breakout_down',
       'AAPL_consolidation', 'AAPL_uptrend', 'AAPL_downtrend',
       'AAPL_overbought', 'AAPL_oversold', 'MSFT_gap_up', 'MSFT_gap_down',
       'MSFT_exhaustion_gap_up', 'MSFT_exhaustion_gap_down', 'MSFT_high_vol',
       'MSFT_extreme_vol', 'MSFT_breakout_up', 'MSFT_breakout_down',
       'MSFT_consolidation', 'MSFT_uptrend', 'MSFT_downtrend',
       'MSFT_overbought', 'MSFT_oversold'],
      dtype='object')